In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

class HeartbeatDataProcessor:
    def __init__(self, folder_path, filtered_df_path,window_size=2, step_size=1,boundary_cut=5):
            """
            Initializes the data pipeline reader.
            
            Parameters:
            - file_path (str): Path to the df_intervals CSV data.
            - window_size (int): Number of consecutive intervals per training chunk.
            - step_size (int): How far the window slides forward (enables overlapping).
            """
            self.folder_path =   folder_path
            self.filtered_df_path = filtered_df_path
            self.window_size = window_size
            self.step_size = step_size
            self.boundary_cut = boundary_cut
            # Initialize internal storage and stateful scaler
            self.df_filtered = None
            self.filtered_index = None
            self.scaler = StandardScaler()

    def load_filtered_df(self,subject_num):
        #the data here already filtered by activity id!=0 and length>20
        filtered_df_path = f"{self.filtered_df_path}filtered_activities.csv"
        self.filtered_index = pd.read_csv(filtered_df_path)

    def preprocess_subjects(self,subject_array):
        # 1. Ensure the filtered index is loaded
        if self.filtered_index is None:
            self.load_filtered_df(subject_array)

        all_chunks = []

        # 2. Preprocess each subject
        for subject_num in subject_array:
            file_path = f"{self.folder_path}subject{subject_num}.dat"
            df_raw = pd.read_csv(file_path, sep='\s+', header=None)
            subject_intervals = self.filtered_index[self.filtered_index['subject_id'] == subject_num]

            for interval_id, interval in subject_intervals.iterrows():
                start_t = int(interval['t1'])+self.boundary_cut
                end_t = int(interval['t2'])-self.boundary_cut

                # 3. Extract the interval from the raw data
                chunk = df_raw[(df_raw[0]>=start_t) & (df_raw[0]<=end_t)].copy()
                if not chunk.empty:
                    chunk['subject_id'] = subject_num
                    chunk['interval_id'] = interval_id

                all_chunks.append(chunk)
                
        # 4. Combine all chunks into a single DataFrame
        self.df_filtered  = pd.concat(all_chunks,ignore_index=True)

        # 5. Standardize the features
        # df_combined[['x', 'y', 'z']] = self.scaler.fit_transform(df_combined[['x', 'y', 'z']])

        def extract_features(self,window_df):
            # Extract features from the windowed data
            return self

    # def standardize(self, columns):
    #     # Standardize features (mean=0, std=1) 
    #     # Crucial: only fit on the training portion if this were a full pipeline
    #     self.df[columns] = self.scaler.fit_transform(self.df[columns])
    #     return self

    # def segment(self, window_size):
    #     # Basic sliding window segmentation
    #     segments = []
    #     for i in range(len(self.df) - window_size + 1):
    #         segments.append(self.df.iloc[i : i + window_size])
    #     return segments

In [3]:
folder_path='../data/PAMAP2_Dataset/Protocol/'
filtered_df_path='./'
processed_data = HeartbeatDataProcessor(folder_path,filtered_df_path)
processed_data.preprocess_subjects(range(101,108))
print(processed_data.df_filtered.head(3))

       0  1   2        3         4        5        6         7        8  \
0  42.00  1 NaN  30.4375 -0.838906  9.39300  4.42189 -0.570975  9.60023   
1  42.01  1 NaN  30.4375 -0.662789  9.08535  4.11766 -0.695081  9.35932   
2  42.02  1 NaN  30.4375 -0.956803  8.55955  4.46142 -0.865294  8.95268   

         9  ...        46       47       48       49   50   51   52   53  \
0  4.77537  ... -0.018974 -57.0102 -42.1745 -59.8213  1.0  0.0  0.0  0.0   
1  4.59481  ... -0.036326 -56.7818 -42.6205 -60.0755  1.0  0.0  0.0  0.0   
2  4.47507  ... -0.016646 -56.0937 -43.5321 -59.7195  1.0  0.0  0.0  0.0   

   subject_id  interval_id  
0         101            0  
1         101            0  
2         101            0  

[3 rows x 56 columns]
